In [12]:
def rule(header):
    """Separator for a '|'-joined header: '+' under every '|', '-' everywhere else."""
    start = max(len(header) - len(header.lstrip()), 0)
    return " " * start + "".join("+" if c == "|" else "-" for c in header[start:])
def table_header(header):
    print(header)
    print(rule(header))

def header(columns, debug=False):
    """Header block for an ASCII table: the label rows, then the rule.

    A column is a string, or a tuple of 2+ lines whose extra rows are centered
    under the first. Column width = the widest of that column's own lines.
    debug=True prepends a row showing each column's computed width."""
    cols = [(c,) if isinstance(c, str) else tuple(c) for c in columns]
    widths = [max(map(len, c)) for c in cols]

    lines = []
    for r in range(max(map(len, cols))):
        cells = []
        for col, w in zip(cols, widths):
            text = col[r] if r < len(col) else ""
            cells.append(text.ljust(w) if r == 0 else text.center(w))
        lines.append(" | ".join(cells))

    rule = "".join("+" if ch == "|" else "-" for ch in lines[0])
    if debug:
        lines.insert(0, " | ".join(str(w).center(w) for w in widths))
    lines.append(rule)
    print("\n".join(line.rstrip() for line in lines))


In [13]:
import numpy as np
import gymnasium as gym
DIMS = ["x (cart pos)", "x_dot (cart vel)", "theta (pole angle)", "theta_dot (pole vel)"]
BOX = np.array([[-2.4, 2.4], [-3.0, 3.0], [-0.2095, 0.2095], [-3.0, 3.0]])

In [14]:
def rollout_states(num_episodes=200, seed=0):
    env = gym.make('CartPole-v1')
    rng = np.random.default_rng(seed)
    states, deltas = [], []
    for ep in range(num_episodes):
        s, _ = env.reset(seed = seed + ep)
        done = False
        while not done:
            states.append(s)
            action = int(rng.integers(2))
            ns, _r, term, trunc, _ = env.step(action)
            deltas.append(np.abs(ns - s))
            done = term or trunc
            s = ns
        states.append(s)
    return np.array(states), np.array(deltas)

In [25]:
num_episodes=200
seed=0
states, deltas = rollout_states(num_episodes, seed)
print(f'{states.shape=}, {deltas.shape=}')
print(f'{states[0]=}')
T = len(states)
print(f'{num_episodes} random-policy episodes, {T:,} states visited')

states.shape=(4738, 4), deltas.shape=(4538, 4)
states[0]=array([ 0.01369617, -0.02302133, -0.04590265, -0.04834723], dtype=float32)
200 random-policy episodes, 4,738 states visited


In [16]:
uniq = len(np.unique(states, axis=0))
print(f'distinct raw states: {uniq:,} / {T:,}')
print(f'repeats: {T - uniq}')

distinct raw states: 4,738 / 4,738
repeats: 0


In [17]:
env = gym.make('CartPole-v1')
space = env.observation_space
table_header(f'{'dim':22s} | observed min | observed max | env bounds')

for i, name in enumerate(DIMS):
    bound = f'[{space.low[i]:.2f}, {space.high[i]:.2f}]'
    print(
        f"{name:22s} | {states[:, i].min():^+12.3f} | {states[:, i].max():^+12.3f}"
        f" | {bound}"
    )

dim                    | observed min | observed max | env bounds
-----------------------+--------------+--------------+-----------
x (cart pos)           |    -0.381    |    +0.439    | [-4.80, 4.80]
x_dot (cart vel)       |    -1.977    |    +1.982    | [-inf, inf]
theta (pole angle)     |    -0.250    |    +0.252    | [-0.42, 0.42]
theta_dot (pole vel)   |    -3.051    |    +3.070    | [-inf, inf]


In [18]:
def to_cell(states, bins):
    lo, hi = BOX[:, 0], BOX[:, 1]
    idx = ((states - lo) / (hi - lo) * bins).astype(int)
    idx = np.clip(idx, 0, bins - 1)
    result = np.ravel_multi_index(idx.T, (bins,) * 4)
    return result, idx

In [23]:
# each state as a 4-digit number written in base bins
# for bins=3
# id = i0*bins³ + i1*bins² + i2*bins¹ + i3
# [1,0,2,1] -> 34

c, idx = to_cell(states, bins=3)
print(f'{c=}')
print(f'{idx[0]=}, {c[0]=}')
print(f'{idx[4]=}, {c[4]=}')
print(f'{idx[15]=}, {c[15]=}')

c=array([40, 40, 40, ..., 37, 37, 37], shape=(4738,))
idx[0]=array([1, 1, 1, 1]), c[0]=np.int64(40)
idx[4]=array([1, 1, 0, 1]), c[4]=np.int64(37)
idx[15]=array([1, 1, 0, 0]), c[15]=np.int64(36)


In [24]:
table_header(f"bins/dim | table cells | (s,a) pairs | cells seen | coverage | visits/seen")
for bins in (3, 5, 10, 20, 50):
    cells = bins ** 4
    c, _ = to_cell(states, bins)
    counts = np.bincount(c)
    seen = int((counts > 0).sum())
    # 2 actions per state
    sa = 2 * cells
    print(
        f"{bins:8d} | {cells:11,d} | {sa:11,d} | {seen:10,d}"
        f" | {100 * seen/cells:7.2f}% | {counts[counts > 0].mean():11.1f}"
    )

bins/dim | table cells | (s,a) pairs | cells seen | coverage | visits/seen
---------+-------------+-------------+------------+----------+------------
       3 |          81 |         162 |         14 |   17.28% |       338.4
       5 |         625 |       1,250 |         35 |    5.60% |       135.4
      10 |      10,000 |      20,000 |        206 |    2.06% |        23.0
      20 |     160,000 |     320,000 |        725 |    0.45% |         6.5
      50 |   6,250,000 |  12,500,000 |      2,456 |    0.04% |         1.9


In [21]:
print("one step changes the state by (mean |s' - s| per dim):")

for i, name in enumerate(DIMS):
    print(f'{name:22} {deltas[:, i].mean():.4f}')

one step changes the state by (mean |s' - s| per dim):
x (cart pos)           0.0075
x_dot (cart vel)       0.1950
theta (pole angle)     0.0117
theta_dot (pole vel)   0.2920


In [22]:
header([
    'bins/dim',
    'theta bin width',
    ('steps to cross one theta bin', '(bin width / mean step)'),
    ('mean steps', 'inside a cell'),
])
for bins in (3, 5, 10, 20, 50):
    width = (BOX[2, 1] - BOX[2, 0]) / bins
    cross = width / deltas[:, 2].mean()
    c, _ = to_cell(states, bins)
    changes = int((c[1:] != c[:-1]).sum())
    dwell = len(c) / (changes + 1)
    print(f'{bins:8d} | {width:^15.4f} | {f'{cross:>4.1f}':^28} | {dwell:^13.2f}')

bins/dim | theta bin width | steps to cross one theta bin | mean steps
         |                 |   (bin width / mean step)    | inside a cell
---------+-----------------+------------------------------+--------------
       3 |     0.1397      |             11.9             |     4.37     
       5 |     0.0838      |              7.2             |     2.40     
      10 |     0.0419      |              3.6             |     1.34     
      20 |     0.0209      |              1.8             |     1.01     
      50 |     0.0084      |              0.7             |     1.00     
